# Solutions · Chapter 01-02 · Essential Python for data work

Worked answers with reasoning. E9 is the one to read carefully - the buggy function gives the
**right** answer on this chapter's data, which is exactly what makes that class of bug survive.

Self-contained: run from the top with a fresh kernel.

In [ ]:
readings = [
    {"sensor": "s1", "site": "north", "temp_c": 18.5, "ok": True},
    {"sensor": "s2", "site": "north", "temp_c": 24.1, "ok": True},
    {"sensor": "s3", "site": "south", "temp_c": 31.7, "ok": False},
    {"sensor": "s4", "site": "south", "temp_c": 22.0, "ok": True},
    {"sensor": "s5", "site": "east", "temp_c": 27.3, "ok": True},
    {"sensor": "s6", "site": "east", "temp_c": 19.4, "ok": False},
]
print(len(readings), "readings")

## E1 · List or dictionary?

Choose a **dictionary** when you will look things up by name, and when each item has several
named parts. Choose a **list** when order matters or you will iterate over everything.

From this chapter's data:

- **List is right for `readings` itself.** There is no meaningful key for a reading, order is the
  order they arrived, and every operation walks all of them.
- **Dictionary is right for `flags = {r["sensor"]: r["ok"] for r in readings}`.** The question is
  "is s3 working?", which is a lookup by name, and answering it from the list means scanning it.

**The deeper reason, which matters at scale:** a dictionary lookup takes the same time whether it
holds ten items or ten million; scanning a list takes proportionally longer. If you ever write a
loop inside a loop to match items in one list against another, you have written something that
gets 100x slower when the data gets 10x bigger, and the fix is a dictionary - or, in pandas, a
join.

## E2 · Why `log=[]` leaks

> A default argument is evaluated once, when the `def` line runs - not on each call. So a single
> list object is created at definition time and stored on the function, and every call that does
> not supply its own `log` appends to that same object.

The consequence worth stating: `experiment_a` and `experiment_b` are two *names* for one list, so
"experiment B's results" contains experiment A's.

## E3 · `d["k"]` versus `d.get("k")`

`d["k"]` raises `KeyError` if the key is absent. `d.get("k")` returns `None`, and
`d.get("k", default)` returns whatever you specify.

- Use **`["k"]`** when a missing key means something has gone wrong upstream and you want to hear
  about it immediately - a required column, a field the schema promises.
- Use **`.get(...)`** when absence is expected and has a meaningful substitute - an optional
  field, a lookup that legitimately misses.

**The judgement being trained:** every `.get(key, default)` is a small decision to carry on
quietly. That is sometimes right and sometimes the beginning of a silent wrong answer - the same
decision, at scale, as filling missing values in a dataset, which is 02-04.

## E4 · Slicing, by hand

In [ ]:
xs = [10, 20, 30, 40, 50]
for expr in ("xs[1:3]", "xs[:-1]", "xs[::-1]", "xs[3:1]", "len(xs[2:])"):
    print(f"{expr:<14} -> {eval(expr)}")

| Expression | Result | Why |
|---|---|---|
| `xs[1:3]` | `[20, 30]` | Start at 1, stop **before** 3 |
| `xs[:-1]` | `[10, 20, 30, 40]` | Everything except the last - negative indices count from the end |
| `xs[::-1]` | `[50, 40, 30, 20, 10]` | Step of -1 reverses. The idiomatic reversal |
| `xs[3:1]` | `[]` | Start after stop, so nothing. **No error** |
| `len(xs[2:])` | `3` | Elements 2, 3, 4 |

The one to remember is `xs[3:1]`. An empty result where you expected data does not announce
itself - a loop over it simply does nothing, and the downstream code sees an empty list rather
than a crash. If a step in a pipeline mysteriously produces no output, check the slice bounds
before anything else.

## E5 · What `b = a` copies

In [ ]:
a = [1, 2, 3]
b = a          # a second name for the same list
c = a[:]       # a new list with the same contents
b.append(4)

print("a:", a)
print("b:", b)
print("c:", c)
print("a is b:", a is b, "  a is c:", a is c)

`a` and `b` are both `[1, 2, 3, 4]`; `c` is `[1, 2, 3]`.

`b = a` binds a second name to the same object, so appending through either name changes the one
list. `c = a[:]` builds a new list containing the same items, so it is unaffected.

`a is c` is `False` even though their contents are equal - which is the whole distinction between
`is` (same object) and `==` (equal value).

**The caveat that E5 does not show:** `a[:]` is a *shallow* copy. If `a` contained lists, `c`
would hold references to the same inner lists, and mutating one would still be visible through
the other. That is the trap in the chapter's config example.

## E6 · Mean by site, two ways

In [ ]:
# Version 1: accumulate sums and counts in one pass.
def mean_by_site(records):
    totals, counts = {}, {}
    for r in records:
        totals[r["site"]] = totals.get(r["site"], 0) + r["temp_c"]
        counts[r["site"]] = counts.get(r["site"], 0) + 1
    return {site: round(totals[site] / counts[site], 2) for site in totals}


# Version 2: group first, then average - two clear steps.
def mean_by_site_grouped(records):
    grouped = {}
    for r in records:
        grouped.setdefault(r["site"], []).append(r["temp_c"])
    return {site: round(sum(v) / len(v), 2) for site, v in grouped.items()}


print(mean_by_site(readings))
print(mean_by_site_grouped(readings))

Both give `{'north': 21.3, 'south': 26.85, 'east': 23.35}`.

**Version 1** makes one pass and keeps two small dictionaries - memory-efficient, and the shape
you want if the data is too large to hold grouped.

**Version 2** builds the groups first, then reduces them. `setdefault(key, []).append(x)` is the
idiom: get the list for this key, creating an empty one if needed, then append. It is longer in
memory and much easier to extend - once you have the groups you can take the max, the median, the
count, or all of them, without touching the accumulation code.

**Version 2 is also what `groupby` is.** Split into groups, then apply a function to each. When
you write `df.groupby("site")["temp_c"].mean()` in 01-05, this is what it is doing, and knowing
that makes `.agg(["mean", "max"])` obvious rather than magic.

## E7 · The hottest records

In [ ]:
def worst_n(records, n=2):
    return sorted(records, key=lambda r: r["temp_c"], reverse=True)[:n]


for r in worst_n(readings):
    print(f"{r['sensor']} at {r['temp_c']} °C ({r['site']})")

`sorted(..., key=..., reverse=True)[:n]` - sort by the field, largest first, take the top n.

**Why return the records and not the temperatures:** because the next question is always "which
sensor, and where?". A function that returns bare numbers throws away the identity of the row, and
you end up matching values back to records afterwards - which fails as soon as two rows tie.

This exact function reappears throughout the course as *"show me the rows with the largest
error"*, which is the first move of every error analysis (07-05). Keep the record.

**A note on ties:** Python's sort is stable, so records with equal temperatures keep their
original relative order. That makes the output reproducible, which matters when you report "the
worst five" and someone re-runs it.

## E8 · Adding up booleans

In [ ]:
print("working sensors:", sum(r["ok"] for r in readings))
print("True + True =", True + True, "  bool is a subclass of int:", issubclass(bool, int))

In Python `True` **is** 1 and `False` **is** 0 - `bool` is a subclass of `int` - so summing a
sequence of booleans counts the `True`s. It is a genuinely useful idiom, and the pandas and NumPy
versions (`mask.sum()`) are used constantly in this course to count how many rows match a
condition.

**When relying on it is a bad idea:**

- **When the field might not be a real boolean.** A CSV column read as text contains the *strings*
  `"True"` and `"False"`, and both are truthy - so a filter keeps everything and a sum raises. Data
  loaded from a file is guilty until `.dtypes` proves otherwise (01-04).
- **When missing values are possible.** `None` is not 0 and `sum` will raise; in pandas, `NaN`
  propagates and you get `NaN` instead of a count. What you meant was "count the True ones", and
  what you should say is `sum(1 for r in records if r["ok"] is True)`, which is explicit about
  what happens to the unknowns.
- **When readability matters more than brevity.** `sum(bools)` reads as arithmetic and means
  counting. Being explicit costs nothing.

## E9 · Removing items while looping

In [ ]:
def drop_broken(records):
    for r in records:
        if not r["ok"]:
            records.remove(r)
    return records


batch = [{"id": 1, "ok": True}, {"id": 2, "ok": False},
         {"id": 3, "ok": False}, {"id": 4, "ok": True}]

print("result:", [r["id"] for r in drop_broken(batch)], "   <- id 3 was broken and survived")

**It returns ids 1, 3 and 4 - the broken record 3 is still there.**

**Why.** The loop walks the list by position. At position 1 it finds record 2, removes it, and
everything after shifts left - record 3 is now at position 1. But the loop moves on to position 2,
which now holds record 4. Record 3 was stepped over and never examined.

**Two fixes:**

In [ ]:
records = [{"id": 1, "ok": True}, {"id": 2, "ok": False},
           {"id": 3, "ok": False}, {"id": 4, "ok": True}]

# Fix 1 - build a new list. Almost always the right answer.
kept = [r for r in records if r["ok"]]

# Fix 2 - iterate over a copy, if you truly must modify in place.
records_2 = list(records)
for r in list(records_2):
    if not r["ok"]:
        records_2.remove(r)

print("fix 1:", [r["id"] for r in kept])
print("fix 2:", [r["id"] for r in records_2])

In [ ]:
# And now the same buggy function on the chapter's readings.
chapter_copy = [dict(r) for r in readings]
print("buggy result on `readings`:", [r["sensor"] for r in drop_broken(chapter_copy)])
print("correct answer:            ", [r["sensor"] for r in readings if r["ok"]])

### Why the right answer is worse news

On `readings` the buggy function returns **exactly the right result**.

The two broken sensors, s3 and s6, are not adjacent, so each skip lands on a healthy record that
was going to be kept anyway. The bug is fully present and completely invisible.

That is the pattern worth internalising, and it is not really about lists:

> **A test that passes does not mean the code is correct. It means the code is correct on that
> input.**

Change the data - a new batch where two failures happen to arrive together - and the same code
silently keeps a broken sensor. Nothing about the code changed, nothing was redeployed, and the
output is wrong.

This is why chapter 13-05 argues for tests built from the *shape of the failure* rather than from
one sample of real data, and why "it worked on last month's file" is not evidence.

**The rule:** never modify a list while iterating over it. Build a new one.

## E10 · "Why not use a comprehension for everything?"

> Because a comprehension's job is to make a simple transformation obvious at a glance, and past a
> certain complexity it does the opposite. My threshold is concrete: if it needs more than one
> `for`, or more than one `if`, or a conditional expression in the output part, I write the loop.
> A loop that takes six readable lines is better than a one-liner a colleague has to parse
> character by character at 3am - and it is easier to put a `print` inside when it goes wrong.

**What is being tested:** whether you have a rule you can state, or just a preference. Either
answer ("always comprehensions", "always loops") is worse than a threshold.

## E11 · Twelve configurations without cross-contamination

In [ ]:
import copy

base = {"model": "linear", "features": ["temp_c"], "seed": 0}

# The clearest approach: build each one fresh, stating the difference.
configs = [{**copy.deepcopy(base), "model": m, "seed": s}
           for m in ("linear", "tree", "boosting") for s in range(4)]

configs[0]["features"].append("site")            # touch one...
print("config 0 features:", configs[0]["features"])
print("config 1 features:", configs[1]["features"], "  <- unaffected")
print("base    features:", base["features"], "  <- unaffected")
print("total configs:", len(configs))

**The bug being guarded against:** shared nested objects. `base.copy()` or `{**base}` alone
duplicates the top level, but `features` would still be *the same list* in all twelve
configurations - so appending a feature in one experiment silently changes the other eleven, and
the comparison you spend an afternoon on is meaningless.

`copy.deepcopy` copies all the way down, so each configuration owns its own `features` list.

**The habit worth more than the fix:** when a set of experiments produces suspiciously similar
results, check whether they were genuinely different objects. This is the single most common
reason a hyperparameter sweep shows no variation.

**An alternative worth knowing:** make the shared parts immutable - a tuple instead of a list -
so that an accidental mutation raises instead of leaking. Anything that turns a silent bug into a
loud one is worth the small inconvenience.

## E12 · Flattening nested API data

```
result = []
for order in orders:
    city = order["customer"]["city"]          # nested lookup
    discount = order.get("discount", 0.0)     # default for the missing field
    result.append((order["order_id"], city, discount))
```

Or as one comprehension:

```
result = [(o["order_id"], o["customer"]["city"], o.get("discount", 0.0)) for o in orders]
```

**The two features relied on:** `dict.get(key, default)` for the optional field, and chained
subscripting for the nested one.

**The asymmetry is deliberate and is the point of the exercise.** `discount` is optional, so a
default is right. `customer` and `city` are supposed to always be there, so `["..."]` is right -
if one is missing, that is a broken record and you want a `KeyError` now rather than a `None`
travelling silently into a model three steps later.

**What a careful person adds:** a count of how many orders used the default. If 3 of 10,000
orders are missing a discount, fine. If 4,000 are, the field means something different from what
you assumed - perhaps "no discount recorded" versus "no discount given" - and 0.0 is the wrong
substitute. Counting your defaults is a one-line habit that catches wrong assumptions early, and
it is the beginner version of the missing-data analysis in 02-04.

## E13 · Explaining `tuned = baseline`

> You did not make a second document; you wrote a second name on the same one. Both names point
> at one thing, so editing "tuned" edits "baseline" as well, because they are the same object. It
> is like a shared folder that two people have bookmarked - it looks like each of you has a copy
> until one of you changes something.

(58 words.)

**Where the comparison stops being accurate:** with a shared folder you *know* it is shared - the
whole point of the bookmark is that it is the same folder. Here the code looks exactly like making
a copy, which is why the mistake is so easy to make and so hard to see when reading.

## E14 · Your own `groupby`

In [ ]:
def group_by(records, key_field):
    groups = {}
    for r in records:
        groups.setdefault(r[key_field], []).append(r)
    return groups


groups = group_by(readings, "site")
print({site: [r["sensor"] for r in rows] for site, rows in groups.items()})

means = {site: round(sum(r["temp_c"] for r in rows) / len(rows), 2) for site, rows in groups.items()}
print(means)

Eight lines, and it reproduces E6's answer exactly. Grouping is genuinely this simple: a
dictionary from key to a list of members.

**What pandas gives you that this does not:**

- **Speed at scale.** This loops in Python, one row at a time. `df.groupby(...)` runs the same
  logic in compiled code and is orders of magnitude faster on a million rows.
- **Grouping by several keys at once**, giving a hierarchical index you can slice.
- **Many aggregations in one call:** `.agg(["count", "mean", "std", "max"])` returns a table, where
  yours needs a comprehension per statistic.
- **Missing-value handling with a stated policy** - `NaN` keys are dropped by default, which is a
  decision your version makes silently and differently.
- **A result that is itself a DataFrame**, so it can be sorted, joined, filtered and plotted
  without conversion.

**But knowing this version is the point.** `groupby` is not magic, it is `setdefault` in a loop,
and once you see that, `.agg`, `.transform` and `.apply` become obvious variations - apply a
function to each group and return one value, or the same shape, or anything - instead of three
unrelated methods to memorise.

---

## Where to go next

Back to the chapter for the mastery check and flashcards, then **01-03 · NumPy: arrays, shapes,
and vectorised thinking**, where the loops in this chapter get replaced by operations on whole
arrays.